# Contractive autoencoder para el problema de MNIST

## Cargando el dataset de MNIST

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader,random_split
from torch import nn
import torch.nn.functional as F
import torch.optim as optim

data_dir = 'dataset'

train_dataset = torchvision.datasets.MNIST(data_dir, train=True, download=True)
test_dataset = torchvision.datasets.MNIST(data_dir, train=False, download=True)

train_transform = transforms.Compose([
transforms.ToTensor(),
])

test_transform = transforms.Compose([
transforms.ToTensor(),
])

train_dataset.transform = train_transform
test_dataset.transform = test_transform

m=len(train_dataset)

train_data, val_data = random_split(train_dataset, [int(m-m*0.2), int(m*0.2)])
batch_size = 256

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size)
valid_loader = torch.utils.data.DataLoader(val_data, batch_size=batch_size)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size,shuffle=True)

## Creando el autoencoder

In [ ]:
class Encoder(nn.Module):

	def __init__(self, encoded_space_dim,fc2_input_dim):
		super().__init__()

		self.encoder_cnn = nn.Sequential(
		nn.Conv2d(1, 8, 3, stride=2, padding=1),
		nn.ReLU(True),
		nn.Conv2d(8, 16, 3, stride=2, padding=1),
		nn.BatchNorm2d(16),
		nn.ReLU(True),
		nn.Conv2d(16, 32, 3, stride=2, padding=0),
		nn.ReLU(True)
		)

		self.flatten = nn.Flatten(start_dim=1)

		self.encoder_lin = nn.Sequential(
			nn.Linear(3 * 3 * 32, 128),
			nn.ReLU(True),
			nn.Linear(128, encoded_space_dim)
		)

	def forward(self, x):
		x = self.encoder_cnn(x)
		x = self.flatten(x)
		x = self.encoder_lin(x)
		return x

class Decoder(nn.Module):

	def __init__(self, encoded_space_dim,fc2_input_dim):
		super().__init__()
		self.decoder_lin = nn.Sequential(
			nn.Linear(encoded_space_dim, 128),
			nn.ReLU(True),
			nn.Linear(128, 3 * 3 * 32),
			nn.ReLU(True)
		)

		self.unflatten = nn.Unflatten(dim=1,
		unflattened_size=(32, 3, 3))

		self.decoder_conv = nn.Sequential(
			nn.ConvTranspose2d(32, 16, 3,
			stride=2, output_padding=0),
			nn.BatchNorm2d(16),
			nn.ReLU(True),
			nn.ConvTranspose2d(16, 8, 3, stride=2,
			padding=1, output_padding=1),
			nn.BatchNorm2d(8),
			nn.ReLU(True),
			nn.ConvTranspose2d(8, 1, 3, stride=2,
			padding=1, output_padding=1)
		)

	def forward(self, x):
		x = self.decoder_lin(x)
		x = self.unflatten(x)
		x = self.decoder_conv(x)
		x = torch.sigmoid(x)
		return x

## Definiendo funciones de train, test y visualización



In [ ]:
def train_epoch_cae(encoder, decoder, device, dataloader, loss_fn, optimizer,verbose=False):
  encoder.train()
  decoder.train()
  train_loss = []
  for image_batch, _ in dataloader: # with "_" we just ignore the labels (the second element of the dataloader tuple)
    image_batch = image_batch.to(device)
    image_batch.requires_grad_(True)

    encoded_data = encoder(image_batch)
    decoded_data = decoder(encoded_data)
    loss = loss_fn(image_batch, decoded_data, encoded_data, encoder)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if verbose:
      print('\t partial train loss (single batch): %f' % (loss.data))
    train_loss.append(loss.detach().cpu().numpy())

  return np.mean(train_loss)

In [ ]:
def test_epoch_cae(encoder, decoder, device, dataloader, loss_fn):
	encoder.eval()
	decoder.eval()
	with torch.no_grad():
		conc_out = []
		conc_label = []
		for image_batch, _ in dataloader:
			image_batch = image_batch.to(device)

			encoded_data = encoder(image_batch)
			decoded_data = decoder(encoded_data)
			conc_out.append(decoded_data.cpu())
			conc_label.append(image_batch.cpu())

		conc_out = torch.cat(conc_out)
		conc_label = torch.cat(conc_label)
		val_loss = loss_fn(decoded_data, image_batch)
	return val_loss.data

In [ ]:
def plot_ae_outputs_cae(encoder,decoder,n=10):
	plt.figure(figsize=(16,4.5))
	targets = test_dataset.targets.numpy()
	t_idx = {i:np.where(targets==i)[0][0] for i in range(n)}
	for i in range(n):
		ax = plt.subplot(2,n,i+1)
		img = test_dataset[t_idx[i]][0].unsqueeze(0)
		img = img.to(device)

		encoder.eval()
		decoder.eval()

		with torch.no_grad():
			rec_img = decoder(encoder(img))

		plt.imshow(img.cpu().squeeze().numpy(), cmap='gist_gray')
		ax.get_xaxis().set_visible(False)
		ax.get_yaxis().set_visible(False)
		if i == n//2:
			ax.set_title('Original images')

		ax = plt.subplot(2, n, i + 1 + n)
		plt.imshow(rec_img.cpu().squeeze().numpy(), cmap='gist_gray')
		ax.get_xaxis().set_visible(False)
		ax.get_yaxis().set_visible(False)
		if i == n//2:
			ax.set_title('Reconstructed images')

	plt.subplots_adjust(left=0.1,
					bottom=0.1,
					right=0.7,
					top=0.9,
					wspace=0.3,
					hspace=0.3)
	plt.show()

## Definiendo la función de pérdida del CAE

In [ ]:
def contractive_loss_function2(x, x_hat, encoded, encoder, lam=1e-4):
    """
    Contractive loss: MSE + λ * sum_i ||∇_x h_i||²
    x: input batch
    x_hat: reconstructed batch
    encoded: output of encoder
    lam: regularization strength
    """
    mse = torch.nn.functional.mse_loss(x_hat, x)

    # pesos de la última capa lineal
    W = list(encoder.encoder_lin[2].parameters())[0]
    contractive_term = lam * torch.sum(W**2)

    return mse + contractive_term


def contractive_loss_function(x, x_hat, encoded, encoder, lam=1e-4):
    """
    Contractive loss: MSE + λ * sum_i ||∇_x h_i||²
    x: input batch
    x_hat: reconstructed batch
    encoded: output of encoder
    lam: regularization strength
    """
    mse = torch.nn.functional.mse_loss(x_hat, x)

    contractive_term = 0
    for i in range(encoded.shape[1]):
        # Sum over batch to get scalar output for grad
        grad = torch.autograd.grad(
            encoded[:, i].sum(), x, create_graph=True, retain_graph=True)[0]
        contractive_term += torch.sum(grad ** 2)

    return mse + lam * contractive_term



## Entrenando el autoencoder

In [ ]:
### Define an optimizer (both for the encoder and the decoder!)
lr= 0.001

### Set the random seed for reproducible results
torch.manual_seed(0)

### Initialize the two networks
d = 4

#model = Autoencoder(encoded_space_dim=encoded_space_dim)
encoder = Encoder(encoded_space_dim=d,fc2_input_dim=128)
decoder = Decoder(encoded_space_dim=d,fc2_input_dim=128)
params_to_optimize = [
	{'params': encoder.parameters()},
	{'params': decoder.parameters()}
]

optim = torch.optim.Adam(params_to_optimize, lr=lr, weight_decay=1e-05)

# Check if the GPU is available
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f'Selected device: {device}')

# Move both the encoder and the decoder to the selected device
encoder.to(device)
decoder.to(device)

### Training cycle
num_epochs = 15
history_da={'train_loss':[],'val_loss':[]}

for epoch in range(num_epochs):
	print('EPOCH %d/%d' % (epoch + 1, num_epochs))
	### Training (use the training function)
	train_loss=train_epoch_cae(
		encoder=encoder,
		decoder=decoder,
		device=device,
		dataloader=train_loader,
		loss_fn=contractive_loss_function,
		optimizer=optim)
	### Validation (use the testing function)
	val_loss = test_epoch_cae(
		encoder=encoder,
		decoder=decoder,
		device=device,
		dataloader=valid_loader,
		loss_fn=torch.nn.MSELoss())
	# Print Validationloss
	history_da['train_loss'].append(train_loss)
	history_da['val_loss'].append(val_loss)
	print('\n EPOCH {}/{} \t train loss {:.3f} \t val loss {:.3f}'.format(epoch + 1, num_epochs,train_loss,val_loss))
	plot_ae_outputs_cae(encoder,decoder)

## Mostrando el resultado

In [ ]:
from tqdm import tqdm

latent_coords = np.zeros((len(test_dataset),2))
labels = []
i = 0
for sample in tqdm(test_dataset):
  img = sample[0].unsqueeze(0).to(device)
  label = sample[1]
  # Encode image
  encoder.eval()
  with torch.no_grad():
    encoded_img = encoder(img)
    # Append to list
    encoded_img = encoded_img.flatten().cpu().numpy()
    latent_coords[i,0] = encoded_img[0]
    latent_coords[i,1] = encoded_img[1]
    labels.append(label)
    i+= 1

In [ ]:
scatter = plt.scatter(latent_coords[:, 0], latent_coords[:, 1], c=labels, cmap='jet',s=1)
plt.colorbar(scatter, label='Etiqueta del dígito')
plt.title('Representación 2D de dígitos MNIST usando contractive autoencoder')
plt.xlabel('Coordenada latente 1')
plt.ylabel('Coordenada latente 2')
plt.grid(True)
plt.tight_layout()
plt.show()